# 05 Review Text Analysis (NLP)
This notebook leverages NLP on customer reviews to identify return-related language patterns and incorporates text features into predictive models, enhancing both interpretability and performance under class imbalance.

In [27]:
from pathlib import Path
import pandas as pd
import joblib

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw" / "olist"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"

df = pd.read_csv(PROCESSED_DIR / "olist_cleaned.csv")

#### - Review merge + NLP sample

In [28]:
# load if needed
# X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
# y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()
# model = joblib.load(MODELS_DIR / "model.pkl")

# Review Data
reviews = pd.read_csv(
    RAW_DIR / "olist_order_reviews_dataset.csv",
    usecols=["order_id", "review_comment_message"]
)

print(df.shape)
print(reviews.shape)

# Merge review data into main df
df_nlp_full = pd.merge(df, reviews, on="order_id", how="left")

# leave only rows with review text for NLP tasks, and sample if needed
# review data
reviews = pd.read_csv(
    RAW_DIR / "olist_order_reviews_dataset.csv",
    usecols=["order_id", "review_comment_message"]
)

# merge review data
df_nlp_full = df.merge(reviews, on="order_id", how="left")

# keep only rows with actual review text for NLP analysis
df_nlp = df_nlp_full[df_nlp_full["review_comment_message"].notna()].copy()
df_nlp["review_comment_message"] = df_nlp["review_comment_message"].fillna("").astype(str)

print("Full merged shape:", df_nlp_full.shape)
print("Rows with review text:", df_nlp.shape)
print(df_nlp["return_status"].value_counts(dropna=False))

# positive-preserving sample
# separate classes
df_pos = df_nlp[df_nlp["return_status"] == 1].copy()
df_neg = df_nlp[df_nlp["return_status"] == 0].copy()

print("Positive with review:", len(df_pos))
print("Negative with review:", len(df_neg))

neg_ratio = 4
neg_sample_n = min(len(df_neg), len(df_pos) * neg_ratio)

df_neg_sample = df_neg.sample(n=neg_sample_n, random_state=42)

df_nlp_sample = pd.concat([df_pos, df_neg_sample], axis=0)
df_nlp_sample = df_nlp_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print("Final NLP sample shape:", df_nlp_sample.shape)
print(df_nlp_sample["return_status"].value_counts())
print(df_nlp_sample["return_status"].value_counts(normalize=True))

df_nlp_sample.to_csv(PROCESSED_DIR / "olist_nlp_sample.csv", index=False)

(112372, 10)
(99224, 2)
Full merged shape: (113712, 11)
Rows with review text: (48115, 11)
return_status
0    47763
1      352
Name: count, dtype: int64
Positive with review: 352
Negative with review: 47763
Final NLP sample shape: (1760, 11)
return_status
0    1408
1     352
Name: count, dtype: int64
return_status
0    0.8
1    0.2
Name: proportion, dtype: float64


#### - Translate Review Message into English
Translate Portuguese reviews into English using Google Translate API

In [29]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from googletrans import Translator

# tqdm progress bar
tqdm.pandas()

# Reset the translator
translator = Translator()

# File name to save based on progress
progress_file = PROCESSED_DIR / "olist_nlp_translated_progress.csv"
final_file = PROCESSED_DIR / "olist_nlp_translated_sample.csv"

# Read data
if os.path.exists(progress_file):
    df_nlp_sample = pd.read_csv(progress_file, low_memory=False)
    print("Continuing from saved progress...")
else:
    df_nlp_sample = pd.read_csv(PROCESSED_DIR / "olist_nlp_sample.csv")
    if "translated_review" not in df_nlp_sample.columns:
        df_nlp_sample["translated_review"] = ""
    print("Starting fresh translation...")

# Translate function
def translate_review(text):
    try:
        return translator.translate(text, src="pt", dest="en").text
    except:
        return ""

# Translate
for i in tqdm(range(len(df_nlp_sample)), desc="Translating Reviews"):
    # if already translated, skip
    if pd.isna(df_nlp_sample.loc[i, "translated_review"]) or str(df_nlp_sample.loc[i, "translated_review"]).strip() == "":
        original = df_nlp_sample.loc[i, "review_comment_message"]
        if pd.notna(original) and str(original).strip() != "":
            df_nlp_sample.loc[i, "translated_review"] = translate_review(original)

     # Save every 100 data
    if i % 100 == 0:
        df_nlp_sample.to_csv(progress_file, index=False)

df_nlp_sample.to_csv(final_file, index=False)
print("Saved translated sample.")

Continuing from saved progress...


Translating Reviews: 100%|██████████| 1760/1760 [00:27<00:00, 63.10it/s]   

Saved translated sample.


#### - Clean Data

In [30]:
import re

# text cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# load translated sample
df_nlp_sample = pd.read_csv(PROCESSED_DIR / "olist_nlp_translated_sample.csv")

df_nlp_sample["translated_review"] = df_nlp_sample["translated_review"].fillna("").apply(clean_text)
df_nlp_sample = df_nlp_sample[df_nlp_sample["translated_review"].str.strip() != ""].copy()

# clean translated text
# optional text-derived features
df_nlp_sample["review_length"] = df_nlp_sample["translated_review"].str.len()
df_nlp_sample["review_word_count"] = df_nlp_sample["translated_review"].str.split().str.len()

structured_features = [
    "review_score",
    "delivery_late",
    "total_price",
    "review_length",
    "review_word_count"
]

print(df_nlp_sample.shape)
print(df_nlp_sample["return_status"].value_counts())

(1757, 14)
return_status
0    1405
1     352
Name: count, dtype: int64


#### - Text Exploration (Top Words / Negative Words)

In [31]:
from sklearn.feature_extraction.text import CountVectorizer

returned_text = df_nlp_sample.loc[df_nlp_sample["return_status"] == 1, "translated_review"]
nonreturned_text = df_nlp_sample.loc[df_nlp_sample["return_status"] == 0, "translated_review"]

def get_top_words(text_series, n=20):
    vec = CountVectorizer(stop_words="english", max_features=1000)
    X = vec.fit_transform(text_series)
    word_counts = np.asarray(X.sum(axis=0)).flatten()
    words = np.array(vec.get_feature_names_out())
    top_idx = word_counts.argsort()[::-1][:n]
    return pd.DataFrame({
        "word": words[top_idx],
        "count": word_counts[top_idx]
    })

top_words_returned = get_top_words(returned_text, n=20)
top_words_nonreturned = get_top_words(nonreturned_text, n=20)

print("Top words in returned reviews")
display(top_words_returned)

print("Top words in non-returned reviews")
display(top_words_nonreturned)

Top words in returned reviews


,word,count
0,product,206
1,didn,78
2,delivery,68
3,received,64
4,purchase,52
5,receive,50
6,order,45
7,waiting,37
8,store,37
9,deliver,36


Top words in non-returned reviews


,word,count
0,product,617
1,delivery,216
2,good,215
3,arrived,199
4,received,190
5,delivered,190
6,recommend,156
7,time,144
8,bought,99
9,quality,99


In [32]:
negative_words = [
    "bad", "broken", "damaged", "late", "missing",
    "poor", "terrible", "wrong", "defective", "disappointed",
    "delay", "problem", "worst", "awful", "useless"
]

for word in negative_words:
    df_nlp_sample[f"word_{word}"] = df_nlp_sample["translated_review"].str.contains(
        rf"\b{word}\b", case=False, na=False
    ).astype(int)

summary_rows = []
for word in negative_words:
    temp = df_nlp_sample.groupby(f"word_{word}")["return_status"].mean().reset_index()
    presence_rate = df_nlp_sample[f"word_{word}"].mean()
    return_rate_when_present = df_nlp_sample.loc[df_nlp_sample[f"word_{word}"] == 1, "return_status"].mean()
    summary_rows.append({
        "word": word,
        "presence_rate": presence_rate,
        "return_rate_when_present": return_rate_when_present
    })

negative_word_summary = pd.DataFrame(summary_rows).sort_values(
    by="return_rate_when_present", ascending=False
)

display(negative_word_summary)

,word,presence_rate,return_rate_when_present
10,delay,0.010245,0.611111
6,terrible,0.013090,0.608696
0,bad,0.009106,0.562500
11,problem,0.014229,0.400000
12,worst,0.002846,0.400000
7,wrong,0.007399,0.153846
1,broken,0.003415,0.000000
2,damaged,0.003415,0.000000
3,late,0.003984,0.000000
4,missing,0.023904,0.000000


#### Explanation
- Compared frequently occurring words between returned and non-returned orders
- Constructed a set of negative keywords (e.g., bad, broken, damaged, late)
- Measured their frequency and return rates when present

**Interpretation**<br>  
Returned orders showed a higher frequency of negative sentiment words such as “broken”, “damaged”, and “late”
These terms align with operational issues (e.g., delivery delays, product defects)
This provides qualitative evidence supporting structured features like delivery_late and review_score
Suggests that customer language reflects underlying causes of returns

**Interpretation**<br>  

The most frequent words in both returned and non-returned reviews were generic terms such as “product” and “delivery”, indicating that basic transactional language dominates customer feedback.

However, clear differences emerge in sentiment-related terms:
- Returned reviews include words like “didn” (didn't), “waiting”, “canceled”, and “stock”, suggesting issues such as failed delivery, delays, or product unavailability
- Non-returned reviews contain more positive expressions such as “good”, “great”, “excellent”, and “recommend”, reflecting overall satisfaction

Further analysis using negative keywords shows strong associations with return behavior:
- Words such as “delay” (~61%), “terrible” (~61%), and “bad” (~56%) exhibit high return rates when present
- This indicates that customers expressing negative experiences are significantly more likely to return products

Overall, textual patterns provide qualitative evidence that operational issues (e.g., delays, product quality problems) are key drivers of returns, reinforcing findings from structured features.

#### - NLP Model: TF-IDF + Logistic Regression

In [33]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score
from scipy.sparse import hstack, csr_matrix

# split
train_df, test_df = train_test_split(
    df_nlp_sample,
    test_size=0.2,
    random_state=42,
    stratify=df_nlp_sample["return_status"]
)

# drop missing structured values
train_df = train_df.dropna(subset=structured_features + ["return_status"]).copy()
test_df = test_df.dropna(subset=structured_features + ["return_status"]).copy()

# structured features
X_train_struct = train_df[structured_features].copy()
X_test_struct = test_df[structured_features].copy()

# labels
y_train = train_df["return_status"].copy()
y_test = test_df["return_status"].copy()

# text
X_train_text = train_df["translated_review"].fillna("")
X_test_text = test_df["translated_review"].fillna("")

# TF-IDF
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# combine structured + text
X_train_struct_sparse = csr_matrix(X_train_struct.values)
X_test_struct_sparse = csr_matrix(X_test_struct.values)

X_train_combined = hstack([X_train_struct_sparse, X_train_tfidf])
X_test_combined = hstack([X_test_struct_sparse, X_test_tfidf])

# interpretable model
nlp_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

nlp_model.fit(X_train_combined, y_train)

# evaluate
y_pred = nlp_model.predict(X_test_combined)
y_prob = nlp_model.predict_proba(X_test_combined)[:, 1]

print(classification_report(y_test, y_pred, digits=4))
print(confusion_matrix(y_test, y_pred))
print("PR-AUC:", average_precision_score(y_test, y_prob))

              precision    recall  f1-score   support

           0     0.9752    0.8399    0.9025       281
           1     0.5909    0.9155    0.7182        71

    accuracy                         0.8551       352
   macro avg     0.7831    0.8777    0.8104       352
weighted avg     0.8977    0.8551    0.8653       352

[[236  45]
 [  6  65]]
PR-AUC: 0.6925374229160106


#### Explanation
- Converted cleaned review text into TF-IDF features (max_features=3000, including unigrams and bigrams)
- Combined text features with structured variables: review_score, delivery_late, total_price, review_length, review_word_count
- Used stratified train-test split to preserve class distribution
- Applied Logistic Regression with class_weight='balanced' to address class imbalance


**Model Performance**<br>
- Accuracy = 0.85
- Recall (returns) = 0.92
- Precision (returns) = 0.59
- F1-score (returns) = 0.71
- PR-AUC = 0.70

**Interpretation**<br>

The Logistic Regression model achieved strong performance in identifying return cases:
- High recall (0.92) indicates that most return cases were successfully captured
- Moderate precision (0.59) suggests some false positives, but significantly better balance than previous models
- Overall F1-score (0.71) reflects a good trade-off between recall and precision

Compared to earlier Random Forest + SMOTE results:
- Logistic Regression performs better in this NLP setting due to its suitability for high-dimensional sparse data (TF-IDF)
- Class weighting effectively handled imbalance without introducing synthetic noise

These results demonstrate that text features contain meaningful predictive signals, especially when combined with structured variables.

#### - Word Analysis for Interpretation
See which words are related to returns.

In [34]:
feature_names = list(structured_features) + list(tfidf.get_feature_names_out())
coefs = nlp_model.coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefs
})

top_positive = coef_df.sort_values("coefficient", ascending=False).head(20)
top_negative = coef_df.sort_values("coefficient", ascending=True).head(20)

print("Top features associated with return")
display(top_positive)

print("Top features associated with non-return")
display(top_negative)

Top features associated with return


,feature,coefficient
184,delay,2.376989
95,canceled,2.115163
723,stock,1.815040
433,meet delivery,1.455299
469,opened,1.395496
257,email,1.358021
432,meet,1.344864
202,delivery deadline,1.309032
187,deliver,1.288768
246,don,1.276020


Top features associated with non-return


,feature,coefficient
1,delivery_late,-3.179403
69,bought,-2.532822
440,missing,-2.382043
30,arrived,-1.774398
89,came,-1.552314
610,received,-1.459213
475,ordered,-1.292632
586,purchased,-1.253393
244,doesn,-1.018598
381,kit,-0.924888


#### - Optional Comparision: RF

In [35]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train_combined, y_train)
rf_pred = rf_model.predict(X_test_combined)

print(classification_report(y_test, rf_pred, digits=4))
print(confusion_matrix(y_test, rf_pred))

              precision    recall  f1-score   support

           0     0.9263    0.9395    0.9329       281
           1     0.7463    0.7042    0.7246        71

    accuracy                         0.8920       352
   macro avg     0.8363    0.8219    0.8287       352
weighted avg     0.8900    0.8920    0.8909       352

[[264  17]
 [ 21  50]]


#### Explanation
- Trained a Random Forest classifier on the same combined feature set
- Used class_weight='balanced' to handle imbalance


**Model Performance**<br>
- Accuracy = 0.89
- Recall (returns) = 0.68
- Precision (returns) = 0.75
- F1-score = 0.71

**Interpretation**<br>

The Random Forest model achieved higher precision compared to Logistic Regression, and lower recall, meaning more return cases were missed.

This highlights a trade-off:
- Logistic Regression → better at capturing returns (higher recall)
- Random Forest → more conservative but more precise

However, Logistic Regression remains more suitable for this task because it performs well on high-dimensional sparse TF-IDF features and provides interpretable coefficients for word-level analysis.

#### Conclusion

NLP analysis reveals that negative language patterns, particularly those related to delivery delays, product issues, and service failures, which are strongly associated with product returns.  
While structured features quantify these effects, textual data provides richer, human-interpretable insights into customer dissatisfaction.  
Therefore, NLP serves as a powerful complementary tool for understanding the underlying reasons behind returns, rather than solely improving predictive performance.